# Experimental test 4 Result

* vllm기준으로 accuracy와 f1 score를 테스트
    * fewshot 테스트에 활용할 질문의 개수  : 30
    * 소스코드 포함여부  : 'Y'           
    * 반복횟수 : 100회                
    * 시스템프롬프트 'sys_prompt10'
    * self-consistency 횟수 : 5
    * temperature : 0.01
    * 엑셀버전 : 'ver7'
* 이후 결과에 대해서 스코어 비교 진행 


In [1]:
import os
import pandas as pd
from config import config as conf
import re
import numpy as np
from sklearn import metrics



In [2]:
def sc_calc_acc_condition_with_temp_with_sc(llm_model, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    df_eval = pd.DataFrame()
    acc_list = []
    path = f'{conf.DATA_PATH}/{conf.ANNO_RESULT}'
    file_list = os.listdir(path)
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')]

    df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['gold'] = tmp['answer'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp['o_result'] = tmp['result'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp = tmp[tmp['o_result'].isin(['1', '0', '2'])]

            
            gold_df = tmp[['id', 'gold']].drop_duplicates()
            chk_cnt = tmp.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
            chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
            chk_cnt = chk_cnt[chk_cnt['cnt'] == sc_num]
            chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
            df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

            # print(f'size of the dataset : {df_eval.shape[0]}')
            df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
            acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, df_eval], axis =0)
            
        df['equal_yn'] = np.where(df['gold']==df['o_result'], 1, 0)
        y_true = df['o_result']
        y_pred = df['gold']
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list


In [3]:
    # task('vq',              # llm_model
    #     3,                # few_shot_n
    #     60,                # test_n(# of question for test)
    #     'Y',              # q_src_yn 
    #     5,                # iteration num
    #     'sys_prompt10',   # prompt ver
    #     5,                # self-consistency number
    #     0.01,             # temperature
    #     'ver7'            # excel_verion
    #     )

In [4]:

list_ =         sc_calc_acc_condition_with_temp_with_sc('vq', 3, 60, 'Y', 5, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

              precision    recall  f1-score   support

           0      0.960     0.847     0.900        85
           1      0.848     0.884     0.866        95
           2      0.750     0.923     0.828        26

    accuracy                          0.874       206
   macro avg      0.853     0.885     0.865       206
weighted avg      0.882     0.874     0.875       206

vq_result_3_60_Y :  87.37864077669903
[np.float64(87.2340425531915), np.float64(83.72093023255815), np.float64(88.37209302325581), np.float64(87.17948717948718), np.float64(91.17647058823529)]


In [5]:
# sc_vl_result_3_60_Y_100_sys_prompt10_5_0.01_ver7_99.csv
list_ =         sc_calc_acc_condition_with_temp_with_sc('vl', 3, 60, 'Y', 100, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

              precision    recall  f1-score   support

           0      0.962     0.804     0.876      1624
           1      0.726     0.892     0.800      1051
           2      0.875     0.924     0.899       475

    accuracy                          0.851      3150
   macro avg      0.854     0.873     0.858      3150
weighted avg      0.870     0.851     0.854      3150

vl_result_3_60_Y :  85.14285714285714
[np.float64(90.9090909090909), np.float64(82.75862068965517), np.float64(91.66666666666666), np.float64(92.3076923076923), np.float64(88.0), np.float64(83.33333333333334), np.float64(80.0), np.float64(84.61538461538461), np.float64(83.87096774193549), np.float64(80.64516129032258), np.float64(94.11764705882352), np.float64(83.72093023255815), np.float64(82.35294117647058), np.float64(85.0), np.float64(82.14285714285714), np.float64(83.33333333333334), np.float64(87.5), np.float64(87.09677419354838), np.float64(81.81818181818183), np.float64(84.84848484848484), np.float64(77.